# Feature-Level Fusion (Approach b): Concat Autoencoder

**Goal**: replace the raw R-GCN node embedding with the latent code of an autoencoder over `concat(h_graph, h_LLM)`:

$$h_{\text{fused}}(v) = \text{AE}_{\text{enc}}([h_{\text{graph}}(v); h_{\text{LLM}}(v)])$$

**Two phases**:
1. **Phase 1 — Unsupervised AE pretraining.** Reconstruction MSE on the concatenation, computed **only on rows with cached text** (~11.5K nodes), so the AE doesn't waste capacity reconstructing zero placeholders for the other ~117K nodes. R-GCN is frozen here.
2. **Phase 2 — Supervised fine-tuning.** Unfreeze R-GCN + cross-attention + AE encoder (decoder stays frozen). Ranking loss only — no anchoring (per midterm spec).

Run twice: once with `desc_tier="gpt4o"`, once with `desc_tier="hybrid"`.

## 0. Setup

In [ ]:
!pip install -q torch torch-geometric wandb transformers pyyaml matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
PROJECT_ROOT = '/content/drive/MyDrive/BMI Year1/BMI702/project/BMI702'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s: %(message)s')

In [ ]:
import wandb
wandb.login()

## 1. Config

In [ ]:
from pathlib import Path

config = {
    # Data
    'data_dir': 'data/primekg',
    'split_dir': 'data/splits',
    'checkpoint_path': 'data/weights/rgcn_best_model.pt',
    'results_dir': 'results/tables',

    # Text source
    'encoder_name': 'biolinkbert',
    'desc_tier': 'gpt4o',          # change to 'hybrid' for the second run
    'projection': 'nonlinear_ae',
    'embed_dir': 'data/embeddings',

    # Dimensions (locked from checkpoint and cached embeddings)
    'embed_dim': 256,
    'ae_input_dim': 512,           # concat(h_graph[256], h_llm[256])
    'ae_hidden_dim': 512,
    'latent_dim': 256,             # bottleneck back to scorer's expected dim

    # Phase 1 — AE pretraining
    'ae_pretrain_epochs': 50,
    'ae_pretrain_lr': 1e-3,

    # Phase 2 — supervised fine-tune
    'finetune_epochs': 100,
    'patience': 15,
    'lr': 1e-3,
    'weight_decay': 1e-5,
    'margin': 1.0,
    'neg_ratio': 1,
    'accum_steps': 4,
    'grad_clip': 1.0,
    'batch_size': 512,
    'val_frac': 0.1,

    'seed': 42,
    'device': 'cuda',
}

for tier in ('gpt4o', 'hybrid'):
    base = Path(config['embed_dir']) / config['encoder_name'] / tier / config['projection']
    for fname in ('drug_embeddings.pt', 'phenotype_embeddings.pt'):
        p = base / fname
        assert p.exists(), f'Missing {p}'
    print(f'✓ tier={tier}: embeddings present')

assert Path(config['checkpoint_path']).exists()

## 2. Run experiment — Tier: GPT-4o

In [ ]:
from src.models.feature_fusion_train import run_autoencoder_experiment

config['desc_tier'] = 'gpt4o'
results_gpt4o = run_autoencoder_experiment(config)
print('Test (indication-only) MRR :', results_gpt4o['test_metrics_ind']['MRR'])
print('Test (off-label-aug GT) MRR:', results_gpt4o['test_metrics_off']['MRR'])

## 3. Run experiment — Tier: Hybrid

In [ ]:
config['desc_tier'] = 'hybrid'
results_hybrid = run_autoencoder_experiment(config)
print('Test (indication-only) MRR :', results_hybrid['test_metrics_ind']['MRR'])
print('Test (off-label-aug GT) MRR:', results_hybrid['test_metrics_off']['MRR'])

## 4. AE reconstruction-loss curve

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

FIGURES_DIR = Path('results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(results_gpt4o['ae_pretrain_losses'], label='gpt4o', linewidth=2)
ax.plot(results_hybrid['ae_pretrain_losses'], label='hybrid', linewidth=2)
ax.set_xlabel('AE pretrain epoch')
ax.set_ylabel('reconstruction MSE (text-covered rows only)')
ax.set_title('AE Phase-1 reconstruction loss — biolinkbert / nonlinear_ae')
ax.set_yscale('log')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
out = FIGURES_DIR / 'ae_reconstruction_loss.png'
fig.savefig(out, dpi=150)
plt.show()
print(f'Saved {out}')

## 5. gpt4o vs hybrid comparison

In [ ]:
import pandas as pd

METRIC_KEYS = [
    'MRR', 'R@1', 'R@5', 'R@10', 'R@50',
    'AUROC', 'AUPRC',
    'AUROC_balanced_micro', 'AUPRC_balanced_micro',
    'MRR_seen_all', 'MRR_seen_some', 'MRR_seen_none',
]

def metrics_row(tier, gt, m):
    return {'tier': tier, 'gt': gt, **{k: m.get(k, float('nan')) for k in METRIC_KEYS}}

summary = pd.DataFrame([
    metrics_row('gpt4o',  'indication', results_gpt4o['test_metrics_ind']),
    metrics_row('gpt4o',  'off-label',  results_gpt4o['test_metrics_off']),
    metrics_row('hybrid', 'indication', results_hybrid['test_metrics_ind']),
    metrics_row('hybrid', 'off-label',  results_hybrid['test_metrics_off']),
]).set_index(['tier', 'gt'])
summary.round(4)